[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_F.ipynb)


# Set F — Small Networks: STM & WTA (LEGO–Colab)
**Author: Neural Engineering Laboratory, University of Missouri - Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

**Run order:** F0 ▶ F1 ▶ F2 ▶ F3 ▶ F4 ▶ F5.

---
## Table of Contents
- [F0 Starter](#f0)
- [F1 STM (excitatory)](#f1)
- [F2 STM (inhibitory)](#f2)
- [F3 WTA baseline (no inhibition)](#f3)
- [F4 WTA with global inhibition](#f4)
- [F5 Contrast vs decision time](#f5)
- [Reflection](#reflection)
---


## F0 — Starter (run once) <a id='f0'></a>

In [ ]:

!pip -q install neuron==8.2.4 >/dev/null 2>&1
try: from neuron import h, gui
except Exception: from neuron import h
from neuron.units import ms, mV
import numpy as np
import matplotlib.pyplot as plt
h.load_file('stdrun.hoc')

s0=h.Section(name='bootstrap')

def plot_traces(t_vec,Vm_mat,labels,title='',figsize=(7,3)):
    plt.figure(figsize=figsize)
    for vm,l in zip(Vm_mat,labels): plt.plot(t_vec,vm,label=l)
    plt.xlabel('Time (ms)'); plt.ylabel('V (mV)'); plt.title(title)
    if len(labels)<=10: plt.legend(ncol=2,fontsize=8)
    plt.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

def mk_rec(sec): v=h.Vector(); v.record(sec(0.5)._ref_v); return v
print('F0 ready. Proceed to F1.')


In [ ]:

def build_population(N=5,name_prefix='cell'):
    cells=[]
    for i in range(N):
        s=h.Section(name=f"{name_prefix}{i}")
        s.L=s.diam=20; s.Ra=100; s.cm=1; s.insert('hh'); cells.append(s)
    return cells

def connect_all_to_all(cells,kind='E',tau=2.0,e_rev=0.0,weight=0.005,delay=1.0):
    syns=[]; ncs=[]
    for i,pre in enumerate(cells):
        for j,post in enumerate(cells):
            if i==j: continue
            syn=h.ExpSyn(post(0.5)); syn.tau=tau; syn.e=e_rev
            nc=h.NetCon(pre(0.5)._ref_v,syn,sec=pre); nc.threshold=-20; nc.delay=delay; nc.weight[0]=weight
            syns.append(syn); ncs.append(nc)
    return syns,ncs

def brief_iclamp(sec,delay=5,dur=2,amp=0.4): ic=h.IClamp(sec(0.5)); ic.delay=delay; ic.dur=dur; ic.amp=amp; return ic

def tonic_iclamp(sec,amp=0.1,delay=2,dur=1e9): ic=h.IClamp(sec(0.5)); ic.delay=delay; ic.dur=dur; ic.amp=amp; return ic

def run_and_record(cells,tstop=200,vlabels=None):
    t=h.Vector(); t.record(h._ref_t)
    recs=[mk_rec(c) for c in cells]
    h.finitialize(-65*mV); h.continuerun(tstop*ms)
    tnp=np.array(t); vm=[np.array(r) for r in recs]
    if vlabels is None: vlabels=[f"V[{i}]" for i in range(len(cells))]
    plot_traces(tnp,vm,vlabels)
    return tnp,vm

def time_to_win(t,traces):
    t=np.array(t); idx=t>=(t.max()-20)
    means=np.array([np.mean(v[idx]) for v in traces])
    w=int(np.argmax(means))
    V=np.vstack(traces); lead=V[w]-np.max(np.delete(V,w,axis=0),axis=0)
    try: tstar=t[np.where(lead>3)[0][0]]
    except: tstar=np.nan
    return w,tstar,means


## F1 — STM (excitatory all‑to‑all) <a id='f1'></a>

In [ ]:

N=5; cells_E=build_population(N)
synsE,ncsE=connect_all_to_all(cells_E,kind='E',tau=2.0,e_rev=0.0,weight=0.006,delay=1.0)
drive0=brief_iclamp(cells_E[0],delay=5,dur=2,amp=0.4)
t,vm=run_and_record(cells_E,tstop=200,vlabels=[f'E[{i}]' for i in range(N)])
print('Tip: Increase weight to prolong reverberation (STM).')


## F2 — STM (inhibitory all‑to‑all) <a id='f2'></a>

In [ ]:

N=5; cells_I=build_population(N)
synsI,ncsI=connect_all_to_all(cells_I,kind='I',tau=8.0,e_rev=-75.0,weight=0.01,delay=1.0)
drive0=brief_iclamp(cells_I[0],delay=5,dur=2,amp=0.4)
t,vm=run_and_record(cells_I,tstop=200,vlabels=[f'I[{i}]' for i in range(N)])
print('Increase inhibitory weight to quench activity faster.')


## F3 — WTA baseline (excitatory only) <a id='f3'></a>

In [ ]:

N=5; cells_WE=build_population(N)
synsWE,ncsWE=connect_all_to_all(cells_WE,kind='E',tau=2.0,e_rev=0.0,weight=0.003,delay=1.0)
amps=np.linspace(0.08,0.12,N)
ics=[tonic_iclamp(cells_WE[i],amp=float(amps[i]),delay=5.0) for i in range(N)]
t,vm=run_and_record(cells_WE,tstop=400,vlabels=[f'Eonly[{i}]' for i in range(N)])
w,tstar,means=time_to_win(t,vm)
print(f'Baseline (no inhibition): winner={w}, time_to_win≈{tstar}, means={means}')


## F4 — WTA with global inhibition <a id='f4'></a>

In [ ]:

N=5; princ=build_population(N)
inh=h.Section(name='inh'); inh.L=inh.diam=18; inh.Ra=100; inh.cm=1; inh.insert('hh')
amps=np.linspace(0.08,0.12,N)
ics=[tonic_iclamp(princ[i],amp=float(amps[i]),delay=5.0) for i in range(N)]
E2I_syn=[]; E2I_nc=[]
for p in princ:
    syn=h.ExpSyn(inh(0.5)); syn.tau=2.0; syn.e=0.0
    nc=h.NetCon(p(0.5)._ref_v,syn,sec=p); nc.threshold=-20; nc.delay=1.0; nc.weight[0]=0.006
    E2I_syn.append(syn); E2I_nc.append(nc)
I2E_syn=[]; I2E_nc=[]
for p in princ:
    syn=h.ExpSyn(p(0.5)); syn.tau=8.0; syn.e=-75.0
    nc=h.NetCon(inh(0.5)._ref_v,syn,sec=inh); nc.threshold=-20; nc.delay=0.8; nc.weight[0]=0.02
    I2E_syn.append(syn); I2E_nc.append(nc)
_synSE,_ncSE=connect_all_to_all(princ,kind='E',tau=2.0,e_rev=0.0,weight=0.0015,delay=1.0)

t=h.Vector(); t.record(h._ref_t)
recP=[mk_rec(p) for p in princ]; recI=mk_rec(inh)

h.finitialize(-65*mV); h.continuerun(500*ms)
tnp=np.array(t); vP=[np.array(r) for r in recP]; vI=np.array(recI)
plot_traces(tnp,vP,[f'P[{i}]' for i in range(N)],title='F4: WTA with global inhibition')
w,tstar,means=time_to_win(tnp,vP)
print(f'Winner={w}, time_to_win≈{tstar}, final means={means}')


## F5 — Contrast vs decision time <a id='f5'></a>

In [ ]:

import matplotlib.pyplot as plt

def wta_decision_time(contrast=0.04,base=0.08,N=5,inh_w=0.02):
    princ=build_population(N); inh=h.Section(name='inh2'); inh.L=inh.diam=18; inh.Ra=100; inh.cm=1; inh.insert('hh')
    amps=np.linspace(base,base+contrast,N)
    ics=[tonic_iclamp(princ[i],amp=float(amps[i]),delay=5.0) for i in range(N)]
    for p in princ:
        se=h.ExpSyn(inh(0.5)); se.tau=2.0; se.e=0.0
        nc=h.NetCon(p(0.5)._ref_v,se,sec=p); nc.threshold=-20; nc.delay=1.0; nc.weight[0]=0.006
    for p in princ:
        si=h.ExpSyn(p(0.5)); si.tau=8.0; si.e=-75.0
        nc=h.NetCon(inh(0.5)._ref_v,si,sec=inh); nc.threshold=-20; nc.delay=0.8; nc.weight[0]=inh_w
    t=h.Vector(); t.record(h._ref_t)
    recP=[mk_rec(p) for p in princ]
    h.finitialize(-65*mV); h.continuerun(500*ms)
    tnp=np.array(t); vP=[np.array(r) for r in recP]
    w,tstar,means=time_to_win(tnp,vP)
    return w,tstar,means

contrasts=[0.005,0.01,0.02,0.03,0.04]; times=[]
for c in contrasts:
    w,tstar,_=wta_decision_time(contrast=c)
    times.append(tstar)
plt.figure(figsize=(4.8,3)); plt.plot(contrasts,times,'o-')
plt.xlabel('Input contrast (Δamp)'); plt.ylabel('Time‑to‑win (ms)'); plt.title('F5: WTA decision time vs contrast')
plt.grid(True,alpha=0.3); plt.tight_layout(); plt.show()


## Reflection <a id='reflection'></a>
- Why does E‑E feedback support STM? How does leak counteract it?
- Which parameters most strongly shape WTA speed: E→I, I→E, Δinput, or τ?
- How could we add realistic noise and test robustness of decisions?


---

## Practice / Discussion Questions — Set F — Small Networks (STM & WTA) (20)

1) Define **reverberation** and explain how it supports **persistence** in a small network.
2) Minimum conditions (connectivity + kinetics) that favor **STM** without runaway.
3) *Predict*: If NMDA‑like kinetics slow, how does persistence change? Why?
4) Describe a **WTA** network with a shared inhibitory pool; list two parameters controlling **time‑to‑win**.
5) How do small **tonic differences** become categorical outcomes in WTA?
6) Two **metrics** for persistence and one for WTA decision quality.
7) *Scenario*: Persistence collapses when inhibition increases slightly—two hypotheses.
8) Why is **inhibitory timing** still critical in networks?
9) Compare energy/robustness trade‑offs: **synaptic WM** vs **reverberation**.
10) *Design*: A perturbation to disambiguate persistent state vs noise.
11) What is an **attractor** here, and how would you show evidence for it?
12) One **failure mode** signaling need to adjust inhibitory balance.
13) How to test **stability** of a persistent state with parameter sweeps.
14) *Compare*: WTA vs soft‑WTA—what circuit change produces the difference?
15) Simple **readout** to quantify memory length as parameters vary.
16) Why can **adaptation** at single‑cell level influence persistence?
17) A **teaching visualization** to help students “see” WTA unfold.
18) One reason to teach both WM mechanisms (synaptic vs reverberation).
19) A **biologically plausible** way to end persistence without external reset.
20) How Set F positions you for **Set G** behavior mapping.
